# LeetCode 969 — Pancake Sorting bằng Python

Link đề: https://leetcode.com/problems/pancake-sorting/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Two Pointers (hai con trỏ), Greedy (tham lam), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        ...
```

Ghi chú: ta chỉ được dùng một thao tác duy nhất, **lật bánh** (`pancake flip`): chọn `k` rồi **đảo ngược đoạn đầu `arr[0..k-1]`**. Phải trả về **dãy các giá trị `k`** làm mảng được sắp xếp, và đề chấp nhận **mọi dãy hợp lệ** miễn không quá `10 * n` lần lật. Vì có **nhiều đáp án đúng** (đáp án mẫu `[4,2,4,3]` chỉ là một trong số đó), ta **không so sánh với đáp án mẫu** mà dùng một **hàm kiểm tra**: mô phỏng từng lần lật, xem mảng cuối đã sắp xếp chưa và số lần lật có `<= 10 * n` không.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một phép biến đổi duy nhất (đảo đoạn đầu), và đáp án là **dãy thao tác**, không phải mảng đã sắp xếp.
2. Nắm các công cụ Python cần thiết: cắt lát và đảo (`arr[:k][::-1]`), gán lát (`arr[:k] = ...`), `list.index`, `range(n, 1, -1)` đếm ngược, và `collections.deque` cho BFS.
3. Hiểu nguyên lý **chọn phần tử lớn nhất và đặt vào cuối bằng hai lần lật**: lật để đưa nó lên đầu, rồi lật cả đoạn chưa xong để đẩy nó xuống đúng chỗ; sau đó **bỏ phần đã xong** và lặp lại.
4. Đi qua 3 cách: BFS trên không gian hoán vị (tìm dãy lật **ngắn nhất**, chỉ dùng cho `n` nhỏ và làm oracle), thuật toán hai-lần-lật lặp (cách nộp chính), và bản đệ quy kèm dừng sớm khi đã sắp xếp; kiểm tra bằng **hàm mô phỏng**, không so sánh với đáp án mẫu.
5. Tránh các lỗi kinh điển: `k` tính theo **độ dài** (1-indexed) chứ không phải chỉ số, quên lật lên đầu, không thu hẹp phạm vi sau mỗi vòng, quên cập nhật mảng sau khi lật, và tìm phần tử lớn nhất trên cả mảng thay vì phần chưa xong.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`, hãy sắp xếp nó bằng một chuỗi **lật bánh**. Mỗi lần lật:

- chọn số nguyên `k` với `1 <= k <= arr.length`;
- **đảo ngược** đoạn `arr[0...k-1]` (chỉ số bắt đầu từ 0).

Ví dụ `arr = [3,2,1,4]`, lật với `k = 3` thì đảo `[3,2,1]` và `arr = [1,2,3,4]`.

Trả về **mảng các giá trị `k`** ứng với một dãy lật làm `arr` được sắp xếp. Mọi đáp án sắp xếp được `arr` trong **không quá `10 * arr.length` lần lật** đều được chấp nhận.

**Ví dụ 1:** `arr = [3,2,4,1]` -> `[4,2,4,3]` (4 lần lật với `k = 4, 2, 4, 3`; các trạng thái: `[3,2,4,1] -> [1,4,2,3] -> [4,1,2,3] -> [3,2,1,4] -> [1,2,3,4]`).

**Ví dụ 2:** `arr = [1,2,3]` -> `[]` (đã sắp xếp; `[3,3]` cũng được chấp nhận).

**Ràng buộc:**

- `1 <= arr.length <= 100`
- `1 <= arr[i] <= arr.length`
- Các phần tử **khác nhau**: `arr` là một hoán vị của `1..arr.length`.

In [1]:
from typing import List, Optional, Tuple, Dict
from collections import deque
from itertools import permutations
import random, time

def apply_flip(arr: List[int], k: int) -> List[int]:
    """Trả về mảng mới sau khi lật k phần tử đầu (không sửa mảng gốc)."""
    return arr[:k][::-1] + arr[k:]

def check_flips(arr: List[int], flips: List[int]) -> bool:
    """Hàm kiểm tra: mọi k hợp lệ (1..n), số lần lật <= 10 * n, và mô phỏng xong thì mảng đã sắp xếp."""
    n = len(arr)
    if len(flips) > 10 * n:
        return False
    a = list(arr)
    for k in flips:
        if isinstance(k, bool) or not isinstance(k, int) or not 1 <= k <= n:
            return False
        a = apply_flip(a, k)
    return a == sorted(arr)

examples = [([3, 2, 4, 1], [4, 2, 4, 3]), ([1, 2, 3], [])]
for arr, sample in examples:
    print(f"arr={arr} đáp án mẫu={sample} -> hợp lệ theo hàm kiểm tra: {check_flips(arr, sample)}")
    assert check_flips(arr, sample)
print("đáp án khác cũng được:", check_flips([1, 2, 3], [3, 3]), "| đáp án sai:", check_flips([3, 2, 4, 1], [4, 2, 4]))

arr=[3, 2, 4, 1] đáp án mẫu=[4, 2, 4, 3] -> hợp lệ theo hàm kiểm tra: True
arr=[1, 2, 3] đáp án mẫu=[] -> hợp lệ theo hàm kiểm tra: True
đáp án khác cũng được: True | đáp án sai: False


## 3. Kiến thức nền

### 3.1. Đảo một đoạn đầu bằng lát cắt

`arr[:k]` là `k` phần tử đầu; `arr[:k][::-1]` là bản đảo của chúng. Gán ngược lại để sửa tại chỗ: `arr[:k] = arr[:k][::-1]`. Chú ý: `k` là **độ dài** đoạn lật, không phải chỉ số cuối.

In [2]:
arr = [3, 2, 1, 4]
k = 3
print("trước:", arr, "| lật k = 3 ->", apply_flip(arr, k))
a = arr[:]
a[:k] = a[:k][::-1]
print("gán lát tại chỗ:", a)
print("k = 1 không đổi gì:", apply_flip([5, 6, 7], 1), "| k = n đảo cả mảng:", apply_flip([5, 6, 7], 3))

trước: [3, 2, 1, 4] | lật k = 3 -> [1, 2, 3, 4]
gán lát tại chỗ: [1, 2, 3, 4]
k = 1 không đổi gì: [5, 6, 7] | k = n đảo cả mảng: [7, 6, 5]


### 3.2. Lật hai lần: đưa phần tử từ vị trí `i` xuống vị trí cuối

Giả sử phần tử cần đưa xuống cuối đoạn `[0..size-1]` đang ở chỉ số `idx`. Hai lần lật:

1. `k = idx + 1`: đảo `arr[0..idx]`, phần tử đó nhảy lên **đầu** mảng (vì nó ở cuối đoạn vừa đảo);
2. `k = size`: đảo `arr[0..size-1]`, phần tử đầu mảng nhảy xuống **cuối** đoạn, tức vị trí `size - 1`.

In [3]:
a = [3, 2, 4, 1]
size, idx = 4, a.index(4)
print("max =", 4, "ở chỉ số", idx, "| lật k =", idx + 1, "->", apply_flip(a, idx + 1))
a1 = apply_flip(a, idx + 1)
print("lật k =", size, "->", apply_flip(a1, size), "(số 4 đã về cuối)")

max = 4 ở chỉ số 2 | lật k = 3 -> [4, 2, 3, 1]
lật k = 4 -> [1, 3, 2, 4] (số 4 đã về cuối)


### 3.3. Hàm kiểm tra thay vì so sánh đáp án

Có **nhiều dãy lật đúng**, nên ta kiểm tra bằng hàm `check_flips` (đã định nghĩa ở trên): mô phỏng mọi lần lật rồi so sánh với `sorted(arr)`, đồng thời kiểm tra `1 <= k <= n` và số lần lật `<= 10 * n`. Hàm kiểm tra này **độc lập** với mọi thuật toán giải.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `arr = [3,2,4,1]`, `n = 4`.

- Số lớn nhất `4` ở chỉ số 2. Lật `k = 3`: `[4,2,3,1]`; lật `k = 4`: `[1,3,2,4]`. Số `4` đã về đúng chỗ (cuối).
- Còn `[1,3,2]` (bỏ qua `4`). Số lớn nhất `3` ở chỉ số 1. Lật `k = 2`: `[3,1,2,4]`; lật `k = 3`: `[2,1,3,4]`.
- Còn `[2,1]`. Số lớn nhất `2` ở chỉ số 0 (đã ở đầu nên không cần lật lên đầu). Lật `k = 2`: `[1,2,3,4]`. Xong.

Dãy lật `[3,4,2,3,2]` (5 lần). Khác đáp án mẫu `[4,2,4,3]` nhưng **cũng hợp lệ**. Mỗi vòng ta cố định được **một** phần tử ở cuối nên cần tối đa `2 * (n - 1)` lần lật.

## 5. Cách 1 — Brute Force: BFS trên mọi hoán vị (dãy lật ngắn nhất)

**Ý tưởng.** Mỗi mảng là một **đỉnh**; mỗi lần lật `k` là một **cạnh** tới mảng mới. Từ mảng ban đầu chạy **BFS** cho đến khi gặp mảng đã sắp xếp: đường đi ngắn nhất cho **dãy lật ít nhất** (đây là bài "pancake number"). Cách này không cần ý tưởng thông minh, **luôn tìm ra tối ưu**, nhưng số đỉnh là `n!` nên chỉ dùng cho `n` nhỏ. Nó cũng là **oracle**: số lần lật ngắn nhất cho ta biết thuật toán chính còn xa tối ưu bao nhiêu.

In [4]:
def pancake_bfs(arr: List[int]) -> List[int]:
    start, goal = tuple(arr), tuple(sorted(arr))
    if start == goal:
        return []
    n = len(arr)
    parent: Dict[tuple, Optional[Tuple[tuple, int]]] = {start: None}
    dq = deque([start])
    while dq:
        cur = dq.popleft()
        for k in range(2, n + 1):                       # k = 1 là lật vô nghĩa
            nxt = cur[:k][::-1] + cur[k:]
            if nxt in parent:
                continue
            parent[nxt] = (cur, k)
            if nxt == goal:
                path = []
                node = nxt
                while parent[node] is not None:
                    node, k2 = parent[node]
                    path.append(k2)
                return path[::-1]
            dq.append(nxt)
    raise AssertionError("không thể xảy ra với hoán vị")

for arr, sample in examples:
    res = pancake_bfs(arr)
    print(f"arr={arr} -> BFS ngắn nhất {res} ({len(res)} lần lật), đáp án mẫu dài {len(sample)}")
    assert check_flips(arr, res) and len(res) <= len(sample)

arr=[3, 2, 4, 1] -> BFS ngắn nhất [2, 3, 4] (3 lần lật), đáp án mẫu dài 4
arr=[1, 2, 3] -> BFS ngắn nhất [] (0 lần lật), đáp án mẫu dài 0


**Độ phức tạp và nút thắt.** Có tới `n!` đỉnh, mỗi đỉnh `n - 1` cạnh, mỗi bước tạo mảng mới `O(n)`: `O(n! * n^2)`. Đo thử:

In [5]:
rng = random.Random(969)
for n in (5, 6, 7, 8):
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    t0 = time.perf_counter()
    res = pancake_bfs(arr)
    dt = time.perf_counter() - t0
    print(f"n={n}: {dt * 1000:9.1f} ms, ngắn nhất {len(res)} lần lật")
print("(mỗi n: một hoán vị ngẫu nhiên có seed; BFS tăng rất nhanh theo n!)")
print("n = 100 sẽ cần duyệt tới 100! ~ 9.3 * 10^157 hoán vị: không thể")

n=5:       0.0 ms, ngắn nhất 4 lần lật
n=6:       0.1 ms, ngắn nhất 4 lần lật
n=7:       2.4 ms, ngắn nhất 6 lần lật
n=8:       1.3 ms, ngắn nhất 5 lần lật
(mỗi n: một hoán vị ngẫu nhiên có seed; BFS tăng rất nhanh theo n!)
n = 100 sẽ cần duyệt tới 100! ~ 9.3 * 10^157 hoán vị: không thể


Nút thắt: ta không cần **tối ưu**, đề chỉ cần `<= 10 * n` lần lật. Vậy có một thuật toán **tham lam** gọn dùng `~ 2n` lần lật.

## 6. Ý tưởng thuật toán / bước đột phá

**Giống sắp xếp chọn (selection sort), nhưng "chọn" bằng hai lần lật.** Với `size = n, n - 1, ..., 2`:

1. tìm vị trí `idx` của số **lớn nhất** trong `arr[0..size-1]`; vì `arr` là hoán vị của `1..n` và các giá trị `size+1..n` đã nằm đúng chỗ ở cuối, số lớn nhất của đoạn chính là **giá trị `size`**;
2. nếu nó đã ở cuối (`idx == size - 1`): không làm gì;
3. nếu không: lật `k = idx + 1` đưa nó lên đầu (bỏ qua nếu `idx == 0`), rồi lật `k = size` đẩy nó xuống chỗ cuối của đoạn.

Sau mỗi vòng, phần đuôi `arr[size-1..]` là `size, size+1, ..., n` đã đúng. Mỗi vòng tốn tối đa 2 lần lật nên tổng `<= 2 * (n - 1) <= 198 <= 10 * n`.

| Cách | Ý tưởng |
|---|---|
| 2 | Lặp: tìm số lớn nhất của đoạn, hai lần lật, thu hẹp đoạn (nộp) |
| 3 | Cùng ý tưởng nhưng viết **đệ quy** và **dừng sớm** khi phần còn lại đã sắp xếp |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Lặp: hai lần lật mỗi phần tử (cách nộp chính)

**Giải thích từng đoạn.**

- `a = list(arr)`: sao chép để không sửa mảng đầu vào.
- `for size in range(n, 1, -1)`: đoạn đang xử lý là `a[0..size-1]`; dừng ở `size = 2` vì phần tử cuối cùng còn lại tự đúng chỗ.
- `idx = a.index(size)`: vị trí của giá trị `size`; do `a` là hoán vị của `1..n` và phần `> size` đã ở đúng chỗ, giá trị `size` luôn nằm trong `a[0..size-1]`.
- `idx == size - 1`: đã đúng chỗ, bỏ qua.
- `idx > 0`: lật `k = idx + 1` (đưa lên đầu) và **cập nhật `a`**.
- Lật `k = size` (đẩy xuống cuối đoạn) và cập nhật `a`.

In [6]:
def pancake_iter(arr: List[int]) -> List[int]:
    a = list(arr)
    ans: List[int] = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        if idx == size - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

for arr, sample in examples:
    res = pancake_iter(arr)
    print(f"arr={arr} -> {res}", "hợp lệ" if check_flips(arr, res) else "SAI")
    assert check_flips(arr, res)

arr=[3, 2, 4, 1] -> [3, 4, 2, 3, 2] hợp lệ
arr=[1, 2, 3] -> [] hợp lệ


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        a = list(arr)
        ans = []
        for size in range(len(a), 1, -1):
            idx = a.index(size)
            if idx == size - 1:                 # đã đúng chỗ
                continue
            if idx > 0:                         # đưa lên đầu (k = độ dài đoạn lật = idx + 1)
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)                    # đẩy xuống vị trí size - 1
            a[:size] = a[:size][::-1]
        return ans

sol_iter = Solution().pancakeSort
for arr, _ in examples:
    assert check_flips(arr, sol_iter(list(arr)))
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `n - 1` vòng, mỗi vòng `index` và hai lần đảo lát cắt `O(n)`: thời gian `O(n^2)`; bộ nhớ `O(n)` (bản sao `a` và đáp án). Số lần lật `<= 2 * (n - 1)`.

---

### 7.2. Cách 3 — Đệ quy và dừng sớm

**Ý tưởng khác.** Viết lại bài toán dưới dạng **thu nhỏ**: `solve(size)` đưa giá trị `size` về cuối đoạn rồi gọi `solve(size - 1)`. Thêm một tối ưu: nếu `a[:size]` đã **sắp xếp tăng** thì dừng ngay (không còn phải lật gì), nhờ đó vòng lặp **dừng sớm** trên các mảng gần sắp xếp. (Dãy lật thu được **giống hệt** Cách 2, vì Cách 2 cũng bỏ qua những giá trị đã đúng chỗ; Cách 3 chỉ tiết kiệm phần tìm kiếm còn lại.)

**Giải thích từng đoạn.**

- Điều kiện dừng: `size <= 1` hoặc `a[:size]` đã tăng dần.
- Vòng còn lại giống Cách 2 (tìm `idx`, lật lên đầu nếu cần, lật xuống cuối).
- Đệ quy sâu tối đa `n <= 100`: nằm trong giới hạn mặc định của Python (`1000`).

In [8]:
def pancake_rec(arr: List[int]) -> List[int]:
    a = list(arr)
    ans: List[int] = []

    def solve(size: int) -> None:
        if size <= 1 or all(a[i] < a[i + 1] for i in range(size - 1)):
            return
        idx = a.index(size)
        if idx != size - 1:
            if idx > 0:
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)
            a[:size] = a[:size][::-1]
        solve(size - 1)

    solve(len(a))
    return ans

for arr, _ in examples:
    res = pancake_rec(arr)
    print(f"arr={arr} -> {res}", "hợp lệ" if check_flips(arr, res) else "SAI")
    assert check_flips(arr, res)

arr=[3, 2, 4, 1] -> [3, 4, 2, 3, 2] hợp lệ
arr=[1, 2, 3] -> [] hợp lệ


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        a = list(arr)
        ans = []

        def solve(size: int) -> None:
            if size <= 1 or all(a[i] < a[i + 1] for i in range(size - 1)):
                return                              # đoạn đầu đã sắp xếp: không cần lật nữa
            idx = a.index(size)
            if idx != size - 1:
                if idx > 0:
                    ans.append(idx + 1)
                    a[:idx + 1] = a[:idx + 1][::-1]
                ans.append(size)
                a[:size] = a[:size][::-1]
            solve(size - 1)

        solve(len(a))
        return ans

sol_rec = Solution().pancakeSort
for arr, _ in examples:
    assert check_flips(arr, sol_rec(list(arr)))
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n^2)` (mỗi mức thêm một lần kiểm tra tăng dần `O(n)`); bộ nhớ `O(n)` cho ngăn xếp đệ quy; số lần lật `<= 2 * (n - 1)` và cùng dãy lật với Cách 2.

## 8. TRACE từng bước

### 8.1. Cách 2 trên `[3, 2, 4, 1]`

In mảng sau từng lần lật để thấy phần đuôi đã sắp xếp lớn dần.

In [10]:
def trace_iter(arr):
    a = list(arr)
    ans = []
    print("bắt đầu:", a)
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        print(f"-- size={size}: giá trị {size} ở chỉ số {idx}")
        if idx == size - 1:
            print("   đã đúng chỗ, bỏ qua")
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
            print(f"   lật k={idx + 1} (lên đầu): {a}")
        ans.append(size)
        a[:size] = a[:size][::-1]
        print(f"   lật k={size} (xuống cuối): {a}")
    return ans, a

ans, final = trace_iter([3, 2, 4, 1])
print("dãy lật:", ans, "| mảng cuối:", final)
assert final == [1, 2, 3, 4] and check_flips([3, 2, 4, 1], ans)

bắt đầu: [3, 2, 4, 1]
-- size=4: giá trị 4 ở chỉ số 2
   lật k=3 (lên đầu): [4, 2, 3, 1]
   lật k=4 (xuống cuối): [1, 3, 2, 4]
-- size=3: giá trị 3 ở chỉ số 1
   lật k=2 (lên đầu): [3, 1, 2, 4]
   lật k=3 (xuống cuối): [2, 1, 3, 4]
-- size=2: giá trị 2 ở chỉ số 0
   lật k=2 (xuống cuối): [1, 2, 3, 4]
dãy lật: [3, 4, 2, 3, 2] | mảng cuối: [1, 2, 3, 4]


### 8.2. Hàm kiểm tra mô phỏng đáp án mẫu

Kiểm tra đáp án mẫu `[4,2,4,3]` của đề bằng cách mô phỏng từng lần lật (khớp bảng trạng thái trong đề).

In [11]:
a = [3, 2, 4, 1]
states = [a]
for k in [4, 2, 4, 3]:
    a = apply_flip(a, k)
    states.append(a)
    print(f"k={k}: {a}")
assert states == [[3, 2, 4, 1], [1, 4, 2, 3], [4, 1, 2, 3], [3, 2, 1, 4], [1, 2, 3, 4]]

k=4: [1, 4, 2, 3]
k=2: [4, 1, 2, 3]
k=4: [3, 2, 1, 4]
k=3: [1, 2, 3, 4]


### 8.3. Đối chiếu số lần lật: thuật toán so với tối ưu

Với `arr = [2, 3, 1]`: Cách 2 cho `[2, 3]` (2 lần lật) trong khi BFS cũng cho 2: bằng tối ưu. Với `[3, 1, 2]`: Cách 2 cho `[3, 2]`.

In [12]:
for arr in ([2, 3, 1], [3, 1, 2], [1, 3, 2]):
    print(arr, "-> Cách 2:", pancake_iter(arr), "| BFS tối ưu:", pancake_bfs(arr))
    assert check_flips(arr, pancake_iter(arr)) and len(pancake_bfs(arr)) <= len(pancake_iter(arr))

[2, 3, 1] -> Cách 2: [2, 3] | BFS tối ưu: [2, 3]
[3, 1, 2] -> Cách 2: [3, 2] | BFS tối ưu: [3, 2]
[1, 3, 2] -> Cách 2: [2, 3, 2] | BFS tối ưu: [2, 3, 2]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (lặp, hai lần lật mỗi phần tử): ngắn và `O(n^2)`.

```python
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        a = list(arr)
        ans = []
        for size in range(len(a), 1, -1):
            idx = a.index(size)
            if idx == size - 1:                 # đã đúng chỗ
                continue
            if idx > 0:                         # đưa lên đầu (k = độ dài đoạn lật = idx + 1)
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)                    # đẩy xuống vị trí size - 1
            a[:size] = a[:size][::-1]
        return ans
```

In [13]:
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        a = list(arr)
        ans = []
        for size in range(len(a), 1, -1):
            idx = a.index(size)
            if idx == size - 1:                 # đã đúng chỗ
                continue
            if idx > 0:                         # đưa lên đầu (k = độ dài đoạn lật = idx + 1)
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)                    # đẩy xuống vị trí size - 1
            a[:size] = a[:size][::-1]
        return ans

sol = Solution()
for arr, sample in examples:
    got = sol.pancakeSort(list(arr))
    print(f"arr={arr} -> {got} (đáp án mẫu {sample}):", "hợp lệ" if check_flips(arr, got) else "SAI")
    assert check_flips(arr, got)

arr=[3, 2, 4, 1] -> [3, 4, 2, 3, 2] (đáp án mẫu [4, 2, 4, 3]): hợp lệ
arr=[1, 2, 3] -> [] (đáp án mẫu []): hợp lệ


## 10. Vì sao đúng

**Bất biến.** Sau khi xử lý xong giá trị `size` (cuối vòng lặp với biến `size`), đoạn đuôi `a[size-1..n-1] = [size, size+1, ..., n]`.

- **Khởi tạo** (`size = n + 1`, trước vòng đầu): đoạn đuôi rỗng, hiển nhiên đúng.
- **Duy trì.** Tại vòng `size`, đoạn đầu `a[0..size-1]` chứa đúng các giá trị `1..size` (vì các giá trị `size+1..n` đã nằm ở đuôi). Do đó giá trị `size` nằm ở một chỉ số `idx` trong đoạn đó. Hai lần lật chỉ động tới `a[0..size-1]` nên **không phá** đuôi. Lần lật `k = idx + 1` đảo `a[0..idx]`, đưa giá trị `size` (vốn ở chỉ số `idx`) lên chỉ số `0`. Lần lật `k = size` đảo `a[0..size-1]`, đưa giá trị đang ở chỉ số `0` xuống chỉ số `size - 1`. Nếu `idx == size - 1` thì giá trị đã đúng chỗ. Nếu `idx == 0` thì bỏ lần lật thứ nhất vì giá trị đã ở đầu. Vậy đuôi dài thêm một phần tử.
- **Kết thúc.** Sau `size = 2` đuôi là `[2..n]`, phần tử còn lại ở chỉ số 0 phải là `1`. Mảng đã sắp xếp.

**Số lần lật.** Mỗi vòng `<= 2` lần lật, `n - 1` vòng: `<= 2 * (n - 1) < 10 * n`. `k` luôn thuộc `1..n` (`idx + 1 <= size <= n`).

**Cách 3** thêm kiểm tra dừng sớm: khi `a[:size]` đã tăng dần thì (do đuôi cũng đã đúng) cả mảng đã sắp xếp, nên dừng là đúng.

**Cách 1** (BFS) tìm đường đi ngắn nhất trong đồ thị hoán vị: đúng theo định nghĩa BFS, cho **số lần lật nhỏ nhất**.

**Kiểm chứng bằng máy.** Hàm `check_flips` mô phỏng **độc lập** với thuật toán. Ta kiểm **toàn bộ hoán vị** của `1..n` với `n <= 7` (và `n = 8` cho BFS tính bảng khoảng cách), so số lần lật tối ưu với các giá trị "pancake number" đã biết `0, 1, 3, 4, 5, 7, 8, 9`, kiểm cận `2 * (n - 1)`, và thử các hoán vị ngẫu nhiên với `n` tới `100`.

In [14]:
def optimal_table(n: int) -> Dict[tuple, int]:
    """BFS một lần từ mảng đã sắp xếp: mỗi lần lật là phép tự nghịch đảo, nên khoảng cách đối xứng."""
    goal = tuple(range(1, n + 1))
    dist = {goal: 0}
    dq = deque([goal])
    while dq:
        cur = dq.popleft()
        for k in range(2, n + 1):
            nxt = cur[:k][::-1] + cur[k:]
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                dq.append(nxt)
    return dist

known_pancake_numbers = {1: 0, 2: 1, 3: 3, 4: 4, 5: 5, 6: 7, 7: 8, 8: 9}
for n in range(1, 9):
    dist = optimal_table(n)
    assert len(dist) == len(list(permutations(range(n)))) if n < 8 else True
    assert max(dist.values()) == known_pancake_numbers[n], (n, max(dist.values()))
    if n <= 7:
        worst = 0
        for p in permutations(range(1, n + 1)):
            arr = list(p)
            r2, r3 = sol_iter(arr), sol_rec(arr)
            assert check_flips(arr, r2) and check_flips(arr, r3)
            assert r3 == r2 and len(r2) <= max(0, 2 * (n - 1))        # Cách 3 cho cùng dãy lật với Cách 2
            assert dist[p] <= len(r3)                     # không ai tốt hơn tối ưu
            worst = max(worst, len(r2))
        print(f"n={n}: tối ưu tồi nhất = {max(dist.values())}, Cách 2 tồi nhất = {worst} (cận 2(n-1) = {max(0, 2 * (n - 1))})")
    else:
        print(f"n={n}: tối ưu tồi nhất = {max(dist.values())} (chỉ bảng BFS)")

n=1: tối ưu tồi nhất = 0, Cách 2 tồi nhất = 0 (cận 2(n-1) = 0)
n=2: tối ưu tồi nhất = 1, Cách 2 tồi nhất = 1 (cận 2(n-1) = 2)
n=3: tối ưu tồi nhất = 3, Cách 2 tồi nhất = 3 (cận 2(n-1) = 4)
n=4: tối ưu tồi nhất = 4, Cách 2 tồi nhất = 5 (cận 2(n-1) = 6)
n=5: tối ưu tồi nhất = 5, Cách 2 tồi nhất = 7 (cận 2(n-1) = 8)
n=6: tối ưu tồi nhất = 7, Cách 2 tồi nhất = 9 (cận 2(n-1) = 10)
n=7: tối ưu tồi nhất = 8, Cách 2 tồi nhất = 11 (cận 2(n-1) = 12)


n=8: tối ưu tồi nhất = 9 (chỉ bảng BFS)


## 11. Time / Space Complexity

| Cách | Time | Space | Số lần lật |
|---|---|---|---|
| 1. BFS hoán vị | `O(n! * n^2)` | `O(n! * n)` | **ít nhất có thể** |
| 2. Lặp hai lần lật | `O(n^2)` | `O(n)` | `<= 2(n - 1)` |
| 3. Đệ quy + dừng sớm | `O(n^2)` | `O(n)` | `<= 2(n - 1)`, bằng Cách 2 |

Đo thời gian ở `n = 100` (cỡ tối đa), và thống kê số lần lật trung bình:

In [15]:
rng = random.Random(11)
n = 100
perm = list(range(1, n + 1))
rng.shuffle(perm)
for name, fn in (("Cách 2", sol_iter), ("Cách 3", sol_rec), ("nộp", Solution().pancakeSort)):
    t0 = time.perf_counter()
    for _ in range(200):
        res = fn(list(perm))
    dt = (time.perf_counter() - t0) / 200 * 1000
    assert check_flips(perm, res)
    print(f"{name}: n=100, {dt:7.3f} ms, {len(res)} lần lật (<= {10 * n} cho phép, cận 2(n-1) = {2 * (n - 1)})")

near = list(range(1, n + 1))
for _ in range(3):                                   # mảng gần như đã sắp xếp: chỉ hoán đổi vài cặp gần nhau
    i = rng.randrange(0, n - 1)
    near[i], near[i + 1] = near[i + 1], near[i]
print("mảng gần sắp xếp n = 100:", "Cách 2 =", len(sol_iter(list(near))), "lần lật, Cách 3 =", len(sol_rec(list(near))), "lần lật")
assert sol_rec(list(near)) == sol_iter(list(near))        # cùng dãy lật: Cách 3 chỉ dừng sớm, không rút ngắn dãy

rev = list(range(n, 0, -1))
print("mảng đảo ngược n = 100:", len(Solution().pancakeSort(rev)), "lần lật (thật ra 1 lần lật k = 100 đã đủ):", check_flips(rev, [100]))
for size in (10, 30, 100):
    tot2 = tot3 = 0
    for _ in range(200):
        p = list(range(1, size + 1))
        rng.shuffle(p)
        tot2 += len(sol_iter(p))
        tot3 += len(sol_rec(p))
    print(f"n={size:3}: trung bình Cách 2 = {tot2 / 200:6.1f} lần lật, Cách 3 = {tot3 / 200:6.1f} lần lật")

Cách 2: n=100,   0.098 ms, 180 lần lật (<= 1000 cho phép, cận 2(n-1) = 198)
Cách 3: n=100,   0.137 ms, 180 lần lật (<= 1000 cho phép, cận 2(n-1) = 198)


nộp: n=100,   0.093 ms, 180 lần lật (<= 1000 cho phép, cận 2(n-1) = 198)
mảng gần sắp xếp n = 100: Cách 2 = 12 lần lật, Cách 3 = 12 lần lật
mảng đảo ngược n = 100: 1 lần lật (thật ra 1 lần lật k = 100 đã đủ): True
n= 10: trung bình Cách 2 =   11.9 lần lật, Cách 3 =   11.9 lần lật
n= 30: trung bình Cách 2 =   48.7 lần lật, Cách 3 =   48.7 lần lật
n=100: trung bình Cách 2 =  185.6 lần lật, Cách 3 =  185.6 lần lật


## 12. Bộ test

Vì có **nhiều đáp án hợp lệ**, mọi test dùng `check_flips` (mô phỏng lật, kiểm tra đã sắp xếp, `k` hợp lệ, số lần lật `<= 10 * n`). Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm), toàn bộ hoán vị nhỏ, hoán vị ngẫu nhiên có seed và các tính chất.

In [16]:
def all_solutions():
    return {"iter": pancake_iter, "rec": pancake_rec, "sol_iter": sol_iter, "sol_rec": sol_rec, "final": Solution().pancakeSort}

def check_all(arr, with_bfs=None):
    n = len(arr)
    for name, fn in all_solutions().items():
        orig = list(arr)
        res = fn(list(arr))
        assert check_flips(orig, res), (name, arr, res)
        assert len(res) <= max(0, 2 * (n - 1)), (name, arr, res)
    if with_bfs is None:
        with_bfs = n <= 7
    if with_bfs:
        best = pancake_bfs(list(arr))
        assert check_flips(arr, best)
        assert len(best) <= len(pancake_iter(list(arr)))

# 12.1 ví dụ đề (kể cả đáp án mẫu của đề cũng phải qua hàm kiểm tra)
for arr, sample in examples:
    assert check_flips(arr, sample)
    check_all(arr)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (hàm kiểm tra tự định nghĩa đáp án; kèm kỳ vọng viết tay về số lần lật tối thiểu của Cách 2)
edge = [
    ([1], 0),                         # n = 1: không cần lật
    ([1, 2], 0),                      # đã sắp xếp
    ([2, 1], 1),                      # size = 2, idx = 0: chỉ lật k = 2
    ([1, 2, 3, 4, 5], 0),             # đã sắp xếp: không lật gì
    ([3, 2, 1], 1),                   # size 3: idx 0 -> chỉ lật k = 3 ra [1,2,3]; size 2 đúng chỗ
    ([2, 3, 1], 2),                   # size 3: idx 1 -> k = 2, k = 3 ra [1,2,3]; size 2 đúng chỗ
    ([1, 3, 2], 3),                   # size 3: idx 1 -> k = 2, k = 3 ra [2,1,3]; size 2: idx 0 -> k = 2
    ([5, 4, 3, 2, 1], 1),             # đảo ngược: một lần k = 5
    ([2, 1, 4, 3], None),
    (list(range(1, 101)), 0),         # n = 100, đã sắp xếp
    (list(range(100, 0, -1)), None),  # n = 100, đảo ngược
]
for arr, expected_len in edge:
    check_all(arr, with_bfs=len(arr) <= 7)
    if expected_len is not None:
        got = pancake_iter(list(arr))
        assert len(got) == expected_len, (arr, got, expected_len)
        print(f"  {arr[:6]}{'...' if len(arr) > 6 else ''}: Cách 2 -> {got[:8]}{'...' if len(got) > 8 else ''} ({len(got)} lần lật)")
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
  [1]: Cách 2 -> [] (0 lần lật)
  [1, 2]: Cách 2 -> [] (0 lần lật)
  [2, 1]: Cách 2 -> [2] (1 lần lật)
  [1, 2, 3, 4, 5]: Cách 2 -> [] (0 lần lật)
  [3, 2, 1]: Cách 2 -> [3] (1 lần lật)
  [2, 3, 1]: Cách 2 -> [2, 3] (2 lần lật)
  [1, 3, 2]: Cách 2 -> [2, 3, 2] (3 lần lật)
  [5, 4, 3, 2, 1]: Cách 2 -> [5] (1 lần lật)
  [1, 2, 3, 4, 5, 6]...: Cách 2 -> [] (0 lần lật)
12.2 edge case: OK 11 ca


Ở trên ta kiểm kỳ vọng viết tay về **số lần lật** của Cách 2 (đã tự lần theo thuật toán). Với đáp án, ta luôn dùng **hàm kiểm tra** thay vì so sánh dãy cụ thể, vì dãy lật đúng không duy nhất. Mô phỏng vài ca nhỏ:

In [17]:
for arr in ([3, 2, 1], [2, 3, 1], [1, 3, 2]):
    res = Solution().pancakeSort(list(arr))
    print(arr, "->", res, "| mô phỏng:", end=" ")
    a = list(arr)
    for k in res:
        a = apply_flip(a, k)
    print(a)
    assert a == sorted(arr) and check_flips(arr, res)

[3, 2, 1] -> [3] | mô phỏng: [1, 2, 3]
[2, 3, 1] -> [2, 3] | mô phỏng: [1, 2, 3]
[1, 3, 2] -> [2, 3, 2] | mô phỏng: [1, 2, 3]


In [18]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên mọi hoán vị nhỏ; hàm kiểm tra phát hiện sai)
def wrong_k_is_index(arr):                    # dùng chỉ số idx thay vì độ dài idx + 1
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        if idx == size - 1:
            continue
        ans.append(idx)
        a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

def wrong_no_front_flip(arr):                 # quên lật lên đầu: chỉ lật k = size
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        if idx == size - 1:
            continue
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

def wrong_no_shrink(arr):                     # không thu hẹp: luôn tìm và đặt giá trị lớn nhất ở cuối cả mảng
    a = list(arr)
    ans = []
    n = len(a)
    for size in range(n, 1, -1):
        idx = a.index(size)
        if idx == n - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(n)                         # sai: phải là size
        a[:n] = a[:n][::-1]
    return ans

def wrong_no_update(arr):                     # quên cập nhật mảng sau khi lật (chỉ ghi k)
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        if idx == size - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)               # a không đổi
        ans.append(size)
    return ans

def wrong_global_max(arr):                    # tìm vị trí max trên cả mảng thay vì trên a[:size]
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(max(a))                 # sai: luôn lấy số lớn nhất toàn mảng
        if idx == size - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

def find_counter(wrong, max_n=5):
    for n in range(2, max_n + 1):
        for p in permutations(range(1, n + 1)):
            arr = list(p)
            try:
                res = wrong(list(arr))
            except Exception:
                return arr
            if not check_flips(arr, res):
                return arr
    return None

counters = {}
for name, fn in (("k là chỉ số (idx) thay vì độ dài", wrong_k_is_index), ("quên lật lên đầu", wrong_no_front_flip),
                 ("không thu hẹp phạm vi (luôn k = n)", wrong_no_shrink), ("quên cập nhật mảng sau lật", wrong_no_update),
                 ("tìm max trên cả mảng", wrong_global_max)):
    arr = find_counter(fn)
    assert arr is not None, name
    counters[name] = arr
    res = fn(list(arr))
    a = list(arr)
    ok_k = all(1 <= k <= len(arr) for k in res)
    if ok_k:
        for k in res:
            a = apply_flip(a, k)
    print(f"{name:36} arr={arr}: sai -> {res}" + (f" (mô phỏng ra {a})" if ok_k else " (có k ngoài phạm vi 1..n)"))
    assert check_flips(arr, Solution().pancakeSort(list(arr)))
print("12.3 phản ví dụ: OK")

k là chỉ số (idx) thay vì độ dài     arr=[2, 1]: sai -> [0, 2] (có k ngoài phạm vi 1..n)
quên lật lên đầu                     arr=[1, 3, 2]: sai -> [3, 2] (mô phỏng ra [3, 2, 1])
không thu hẹp phạm vi (luôn k = n)   arr=[1, 2, 3]: sai -> [2, 3] (mô phỏng ra [3, 1, 2])
quên cập nhật mảng sau lật           arr=[1, 3, 2]: sai -> [2, 3, 3, 2] (mô phỏng ra [1, 3, 2])
tìm max trên cả mảng                 arr=[1, 2, 3]: sai -> [3, 2] (mô phỏng ra [2, 3, 1])
12.3 phản ví dụ: OK


In [19]:
# 12.4 toàn bộ hoán vị n <= 6 (với BFS tối ưu) và n = 7 (không BFS từng ca): mọi lời giải hợp lệ
for n in range(1, 7):
    for p in permutations(range(1, n + 1)):
        check_all(list(p), with_bfs=(n <= 5))
for p in permutations(range(1, 8)):
    arr = list(p)
    for fn in all_solutions().values():
        assert check_flips(arr, fn(list(arr)))
print("12.4 mọi hoán vị n <= 7: OK")

# 12.5 ngẫu nhiên có seed, n từ 1 đến 100
rng = random.Random(5969)
for _ in range(800):
    n = rng.randint(1, 100)
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    check_all(arr, with_bfs=False)
print("12.5 ngẫu nhiên (800 ca, n tới 100): OK")

# 12.6 dạng đặc biệt, n = 100: đã sắp xếp, đảo ngược, xoay vòng, chẵn lẻ xen kẽ, hầu như đã sắp xếp
n = 100
special = [
    list(range(1, n + 1)), list(range(n, 0, -1)),
    list(range(2, n + 1)) + [1], [n] + list(range(1, n)),
    list(range(1, n + 1, 2)) + list(range(2, n + 1, 2)),
    list(range(n, 0, -2)) + list(range(n - 1, 0, -2)),
    [2, 1] + list(range(3, n + 1)),
    list(range(1, n - 1)) + [n, n - 1],
]
for arr in special:
    assert sorted(arr) == list(range(1, n + 1))
    check_all(arr, with_bfs=False)
print("12.6 dạng đặc biệt (n = 100): OK")

# 12.7 tính chất: không sửa mảng đầu vào; kết quả là list các int; số lần lật là 0 khi và chỉ khi mảng đã sắp xếp (Cách 3)
rng = random.Random(7969)
for _ in range(300):
    n = rng.randint(1, 30)
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    backup = list(arr)
    res = Solution().pancakeSort(arr)
    assert arr == backup and isinstance(res, list) and all(isinstance(k, int) for k in res)
    assert (len(pancake_rec(list(arr))) == 0) == (arr == sorted(arr))
print("12.7 tính chất: OK")

12.4 mọi hoán vị n <= 7: OK


12.5 ngẫu nhiên (800 ca, n tới 100): OK
12.6 dạng đặc biệt (n = 100): OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `k` là chỉ số, không phải độ dài

Đề định nghĩa `k` là **độ dài** đoạn lật (`1 <= k <= n`). Phần tử ở chỉ số `idx` cần lật đoạn `arr[0..idx]` tức `k = idx + 1`. Ghi `idx` làm sai và có thể sinh `k = 0` (không hợp lệ).

In [20]:
arr = counters["k là chỉ số (idx) thay vì độ dài"]
bad = wrong_k_is_index(list(arr))
print("arr =", arr, "| sai ->", bad, "| đúng ->", Solution().pancakeSort(list(arr)))
assert not check_flips(arr, bad) and check_flips(arr, Solution().pancakeSort(list(arr)))

arr = [2, 1] | sai -> [0, 2] | đúng -> [2]


Vì sao sai: `k = idx` đảo thiếu một phần tử, nên số lớn nhất không lên đầu. **Đúng:** `k = idx + 1`.

### Lỗi 2 — Quên lật lên đầu

Chỉ lật `k = size` (đảo cả đoạn) mà không đưa số lớn nhất lên đầu trước thì số lớn nhất không về cuối.

In [21]:
arr = counters["quên lật lên đầu"]
bad = wrong_no_front_flip(list(arr))
print("arr =", arr, "| sai ->", bad, "| đúng ->", Solution().pancakeSort(list(arr)))
assert not check_flips(arr, bad)

arr = [1, 3, 2] | sai -> [3, 2] | đúng -> [2, 3, 2]


Vì sao sai: lật cả đoạn chỉ đưa số ở **đầu** xuống cuối. **Đúng:** lật `k = idx + 1` (lên đầu) rồi `k = size`.

### Lỗi 3 — Không thu hẹp phạm vi sau mỗi vòng

Dùng `k = n` thay vì `k = size` ở lần lật thứ hai sẽ lật cả phần đuôi đã xong, phá hỏng nó.

In [22]:
arr = counters["không thu hẹp phạm vi (luôn k = n)"]
bad = wrong_no_shrink(list(arr))
print("arr =", arr, "| sai ->", bad, "| đúng ->", Solution().pancakeSort(list(arr)))
assert not check_flips(arr, bad)

arr = [1, 2, 3] | sai -> [2, 3] | đúng -> []


Vì sao sai: lần lật `k = n` đảo luôn phần đã sắp xếp ở cuối. **Đúng:** lật `k = size`, với `size` giảm dần.

### Lỗi 4 — Quên cập nhật mảng sau mỗi lần lật

Chỉ ghi `k` vào đáp án mà không đảo `a` thì các lần `index` sau dùng vị trí cũ, sai hoàn toàn.

In [23]:
arr = counters["quên cập nhật mảng sau lật"]
bad = wrong_no_update(list(arr))
print("arr =", arr, "| sai ->", bad, "| đúng ->", Solution().pancakeSort(list(arr)))
assert not check_flips(arr, bad)

arr = [1, 3, 2] | sai -> [2, 3, 3, 2] | đúng -> [2, 3, 2]


Vì sao sai: các vị trí `idx` phụ thuộc mảng hiện tại. **Đúng:** đảo `a[:k]` ngay khi ghi `k`.

### Lỗi 5 — Tìm số lớn nhất trên cả mảng thay vì đoạn còn lại

`a.index(max(a))` luôn trả về vị trí của `n` (nằm ở đuôi đã xong), không phải số lớn nhất của đoạn chưa xong.

In [24]:
arr = counters["tìm max trên cả mảng"]
bad = wrong_global_max(list(arr))
print("arr =", arr, "| sai ->", bad, "| đúng ->", Solution().pancakeSort(list(arr)))
assert not check_flips(arr, bad)

arr = [1, 2, 3] | sai -> [3, 2] | đúng -> []


Vì sao sai: sau vòng đầu số lớn nhất đã ở cuối và ta cứ lật lại nó. **Đúng:** tìm số lớn nhất **trong `a[:size]`** (ở đây là giá trị `size`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chỉ có một thao tác mạnh nhưng hạn chế (đảo đoạn đầu), cần sắp xếp" ⇒ ý tưởng **chọn rồi cố định phần tử ở cuối**, như selection sort; một phần tử tốn tối đa 2 thao tác.
- **`k` là độ dài**, bắt đầu từ 1; chỉ số 0-indexed ⇒ `k = idx + 1`.
- Đáp án không duy nhất ⇒ **kiểm bằng mô phỏng** (hàm kiểm tra), không so sánh với đáp án mẫu.
- Hoán vị của `1..n` cho phép tìm giá trị `size` bằng `a.index(size)` thay vì `max`.
- Các phép đảo là tự nghịch đảo: lật hai lần cùng `k` trả lại nguyên trạng; nhờ đó BFS từ mảng sắp xếp cho khoảng cách đối xứng.
- Cùng họ: Reverse Subarray To Create Target Array (1460), Minimum Number of K Consecutive Bit Flips (995, cũng là bài về phép lật theo đoạn).
- Test nên có: `n = 1`, đã sắp xếp (đáp án rỗng), đảo ngược, mọi hoán vị nhỏ, `n = 100`.

## 15. Biến thể

1. **Số lần lật ít nhất** (pancake number): BFS cho `n` nhỏ; tìm số lần lật ít nhất cho một mảng cụ thể với `n` lớn là bài toán khó (đã biết là NP-khó); các thuật toán xấp xỉ tốt hơn dùng cỡ `18n/11` lần lật so với `2n` của ta.
2. **Bánh cháy (burnt pancakes):** mỗi bánh có một mặt cháy, cần mặt cháy hướng xuống: dùng ý tưởng tương tự nhưng thêm thao tác lật dấu.
3. **Mảng không phải hoán vị** (có giá trị lặp hoặc bất kỳ): thay `a.index(size)` bằng `a.index(max(a[:size]))` rồi so sánh bằng nhau.
4. Bài liên quan: 1460.

Dưới đây là biến thể 3: giá trị bất kỳ (kể cả lặp). Dùng `max(a[:size])`; kiểm bằng mô phỏng.

In [25]:
def pancake_general(arr: List[int]) -> List[int]:
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        m = max(a[:size])
        idx = max(i for i in range(size) if a[i] == m)     # lần xuất hiện cuối của giá trị lớn nhất (ít lật hơn)
        if idx == size - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

def check_general(arr, flips):
    a = list(arr)
    if len(flips) > 10 * len(a):
        return False
    for k in flips:
        if not 1 <= k <= len(a):
            return False
        a = apply_flip(a, k)
    return a == sorted(arr)

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 12)
    arr = [rng.randint(1, 5) for _ in range(n)]
    assert check_general(arr, pancake_general(arr)), arr
print(pancake_general([3, 1, 3, 2, 1]))

[3, 5, 3, 4]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Số lần lật | Khi nào dùng |
|---|---|---|---|---|---|
| 1. BFS hoán vị | Đường ngắn nhất trong đồ thị hoán vị | `O(n! * n^2)` | `O(n! * n)` | tối ưu | Oracle, `n <= 8` |
| 2. Lặp hai lần lật | Selection sort bằng lật | `O(n^2)` | `O(n)` | `<= 2(n - 1)` | **Cách nộp chính** |
| 3. Đệ quy + dừng sớm | Như Cách 2, dừng khi đã tăng | `O(n^2)` | `O(n)` | `<= 2(n - 1)`, bằng Cách 2 | Dạng đệ quy, dừng sớm |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(apply_flip([4, 1, 3, 2], 3))
print(apply_flip([4, 1, 3, 2], 1))
print(check_flips([2, 1], [2]), check_flips([2, 1], [0]), check_flips([2, 1], [2, 2]))
```

In [26]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None
my_guess_2 = None
my_guess_3 = None

### Bài 2 — Tính tay (dễ-vừa)

Chạy tay Cách 2 trên `arr = [2, 3, 1]` và ghi **dãy `k`** thu được (đúng theo thuật toán trong notebook).

In [27]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành thuật toán lặp.

In [28]:
def exercise_3(arr: List[int]) -> List[int]:
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        # TODO 1: nếu idx == size - 1 thì bỏ qua vòng này
        # TODO 2: nếu idx > 0 thì ghi k = idx + 1 và đảo a[:idx + 1]
        # TODO 3: ghi k = size và đảo a[:size]
        pass
    return ans


# for arr, _ in examples:
#     assert check_flips(arr, exercise_3(arr))

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(arr)` bằng **đệ quy** (`solve(size)`), có dừng sớm khi `a[:size]` đã tăng dần.

In [29]:
def exercise_4(arr: List[int]) -> List[int]:
    # TODO
    pass


# for arr, _ in examples:
#     assert check_flips(arr, exercise_4(arr))

### Bài 5 — Mini challenge: tự viết hàm kiểm tra (khó)

Viết `exercise_5(arr, flips)` trả về `True` nếu `flips` là một đáp án hợp lệ của bài này: mỗi `k` là số nguyên trong `1..n`, số lần lật `<= 10 * n`, và sau khi mô phỏng mọi lần lật mảng được sắp xếp tăng dần.

```python
exercise_5([3, 2, 4, 1], [4, 2, 4, 3]) -> True
exercise_5([1, 2, 3], [])              -> True
exercise_5([1, 2, 3], [3, 3])          -> True
exercise_5([3, 2, 1], [2])             -> False   # chưa sắp xếp
exercise_5([2, 1], [0])                -> False   # k = 0 không hợp lệ
exercise_5([2, 1], [3])                -> False   # k > n
exercise_5([1, 2], [1] * 21)           -> False   # quá 10 * n = 20 lần
exercise_5([1, 2], [1] * 20)           -> True
```
Gợi ý: mô phỏng bằng `a[:k] = a[:k][::-1]` rồi so với `sorted(arr)`.

In [30]:
def exercise_5(arr: List[int], flips: List[int]) -> bool:
    # TODO
    pass


# assert exercise_5([3, 2, 4, 1], [4, 2, 4, 3]) is True
# assert exercise_5([1, 2, 3], []) is True
# assert exercise_5([1, 2, 3], [3, 3]) is True
# assert exercise_5([3, 2, 1], [2]) is False
# assert exercise_5([2, 1], [0]) is False
# assert exercise_5([2, 1], [3]) is False
# assert exercise_5([1, 2], [1] * 21) is False
# assert exercise_5([1, 2], [1] * 20) is True

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `apply_flip(arr, k)` đảo `k` phần tử đầu.
- Mức 2: `[4, 1, 3]` đảo thành `[3, 1, 4]`; `k = 1` không đổi.
- Mức 3: `[3, 1, 4, 2]`, `[4, 1, 3, 2]`, và `True False False` (lật `k = 2` hai lần trả lại `[2, 1]`, chưa sắp xếp).

**Bài 2**
- Mức 1: `size = 3`: giá trị `3` ở chỉ số 1.
- Mức 2: lật `k = 2` (lên đầu) thành `[3, 2, 1]`, lật `k = 3` thành `[1, 2, 3]`.
- Mức 3: `size = 2`: giá trị `2` ở chỉ số 1 (đã đúng chỗ) nên bỏ qua; kết quả `[2, 3]`.

**Bài 3**
- Mức 1: `if idx == size - 1: continue`.
- Mức 2: `if idx > 0: ans.append(idx + 1); a[:idx + 1] = a[:idx + 1][::-1]`.
- Mức 3: `ans.append(size); a[:size] = a[:size][::-1]`.

**Bài 4**
- Mức 1: điều kiện dừng `size <= 1` hoặc đoạn đầu đã tăng.
- Mức 2: thân giống Cách 2, cuối gọi `solve(size - 1)`.
- Mức 3: dùng `all(a[i] < a[i + 1] for i in range(size - 1))` để kiểm tra tăng dần.

**Bài 5**
- Mức 1: kiểm tra độ dài `flips` trước khi mô phỏng.
- Mức 2: từng `k`: `isinstance(k, int)` và `1 <= k <= n`.
- Mức 3: sau khi mô phỏng, `return a == sorted(arr)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [31]:
print(apply_flip([4, 1, 3, 2], 3))
print(apply_flip([4, 1, 3, 2], 1))
print(check_flips([2, 1], [2]), check_flips([2, 1], [0]), check_flips([2, 1], [2, 2]))

assert apply_flip([4, 1, 3, 2], 3) == [3, 1, 4, 2]
assert apply_flip([4, 1, 3, 2], 1) == [4, 1, 3, 2]
assert (check_flips([2, 1], [2]), check_flips([2, 1], [0]), check_flips([2, 1], [2, 2])) == (True, False, False)

[3, 1, 4, 2]
[4, 1, 3, 2]
True False False


### Đáp án Bài 2

`[2, 3, 1]`: `size = 3`, giá trị `3` ở chỉ số 1 -> `k = 2` cho `[3, 2, 1]`, rồi `k = 3` cho `[1, 2, 3]`. `size = 2`: giá trị `2` ở chỉ số 1 (đúng chỗ) nên bỏ qua. Kết quả `[2, 3]`.

In [32]:
hand_answer_solution = [2, 3]
assert Solution().pancakeSort([2, 3, 1]) == hand_answer_solution
assert check_flips([2, 3, 1], hand_answer_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [33]:
def exercise_3_solution(arr: List[int]) -> List[int]:
    a = list(arr)
    ans = []
    for size in range(len(a), 1, -1):
        idx = a.index(size)
        if idx == size - 1:
            continue
        if idx > 0:
            ans.append(idx + 1)
            a[:idx + 1] = a[:idx + 1][::-1]
        ans.append(size)
        a[:size] = a[:size][::-1]
    return ans

for arr, _ in examples:
    assert check_flips(arr, exercise_3_solution(arr))
for n in range(1, 7):
    for p in permutations(range(1, n + 1)):
        assert check_flips(list(p), exercise_3_solution(list(p)))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [34]:
def exercise_4_solution(arr: List[int]) -> List[int]:
    a = list(arr)
    ans = []

    def solve(size: int) -> None:
        if size <= 1 or all(a[i] < a[i + 1] for i in range(size - 1)):
            return
        idx = a.index(size)
        if idx != size - 1:
            if idx > 0:
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)
            a[:size] = a[:size][::-1]
        solve(size - 1)

    solve(len(a))
    return ans

for arr, _ in examples:
    assert check_flips(arr, exercise_4_solution(arr))
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 40)
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    assert check_flips(arr, exercise_4_solution(arr))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [35]:
def exercise_5_solution(arr: List[int], flips: List[int]) -> bool:
    n = len(arr)
    if len(flips) > 10 * n:
        return False
    a = list(arr)
    for k in flips:
        if isinstance(k, bool) or not isinstance(k, int) or not 1 <= k <= n:
            return False
        a[:k] = a[:k][::-1]
    return a == sorted(arr)

assert exercise_5_solution([3, 2, 4, 1], [4, 2, 4, 3]) is True
assert exercise_5_solution([1, 2, 3], []) is True
assert exercise_5_solution([1, 2, 3], [3, 3]) is True
assert exercise_5_solution([3, 2, 1], [2]) is False
assert exercise_5_solution([2, 1], [0]) is False
assert exercise_5_solution([2, 1], [3]) is False
assert exercise_5_solution([1, 2], [1] * 21) is False
assert exercise_5_solution([1, 2], [1] * 20) is True
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 8)
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    flips = [rng.randint(0, n + 1) for _ in range(rng.randint(0, 6))]
    assert exercise_5_solution(arr, flips) == check_flips(arr, flips)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (selection sort bằng lật bánh)

```python
class Solution:
    def pancakeSort(self, arr: List[int]) -> List[int]:
        a = list(arr)
        ans = []
        for size in range(len(a), 1, -1):       # cố định giá trị size ở vị trí size - 1
            idx = a.index(size)
            if idx == size - 1:                 # đã đúng chỗ
                continue
            if idx > 0:                         # lên đầu: k = idx + 1 (độ dài, 1-indexed)
                ans.append(idx + 1)
                a[:idx + 1] = a[:idx + 1][::-1]
            ans.append(size)                    # xuống cuối đoạn
            a[:size] = a[:size][::-1]
        return ans
```

Công thức nhớ nhanh: mỗi giá trị `size` tốn tối đa hai lần lật `k = idx + 1` rồi `k = size`; tổng `<= 2(n - 1)`.

### Checklist trước khi submit

- [ ] `k` là **độ dài** (`idx + 1`), không phải chỉ số; luôn `1 <= k <= n`?
- [ ] Đã **cập nhật mảng** sau mỗi lần lật (đảo `a[:k]`)?
- [ ] Lần lật thứ hai dùng `k = size` (giảm dần), không phải `n`?
- [ ] Tìm giá trị lớn nhất **của đoạn chưa xong** (ở đây là `size`)?
- [ ] Số lần lật `<= 10 * n` (thuật toán dùng `<= 2(n - 1)`)?
- [ ] Kiểm thử bằng **hàm mô phỏng** (không so sánh với đáp án mẫu), gồm `n = 1`, mảng đã sắp xếp, đảo ngược, `n = 100`?